##  Course Details

##### **Programme:** Master of Science in Artificial Intelligence
##### **Course:** CSC 821 - Design and Analysis of Algorithms
##### **Module:** Module 7 - Greedy Algorithms
##### **Assignment:** Lab Activity on Huffman Coding

### Group Details - Group 6 - MSAI

**1. Marrion Kiprop Cherop** — ST62/80971/2024

**2. Kulet Eliud Muchangi** — ST62/56968/2025
  
**3. Michael Musya** — ST62/56377/2025

**4. Mark Makau** — ST62/55795/2025
  
**5. John Obunga Edward** — ST62/55731/2025

**6. Peter Kemari Onyiego** - ST62/55886/2025


### 1. Huffman Coding

Huffman coding is a compression technique that assigns shorter binary codes to characters that appear more frequently and longer codes to characters that appear less often.
The idea is simple, if a character occurs many times, giving it a short code reduces the overall size of the encoded message.

It works by:

1. Counting the frequency of each character in the text.

2. Building a binary tree where low-frequency characters sit deeper and high-frequency characters stay near the root.

3. Generating prefix-free codes (no code is a prefix of another), ensuring the encoded bitstream can be decoded unambiguously.

Huffman coding is widely used in data compression formats such as ZIP files, JPEG, MP3, and many text-based compressors. It is optimal among all prefix-free coding schemes.

In [2]:
import heapq
from typing import Dict, Optional, Tuple


class HuffmanNode:
    def __init__(self, char: Optional[str], freq: int):
        self.char = char      # character (None for internal nodes)
        self.freq = freq      # frequency of this node
        self.left = None      # left child
        self.right = None     # right child

    # Needed so heapq can compare nodes by frequency
    def __lt__(self, other: "HuffmanNode") -> bool:
        return self.freq < other.freq


def build_huffman_tree(freqs: Dict[str, int]) -> HuffmanNode:
    """
    Build Huffman tree from a dict {character: frequency}.
    Returns the root of the Huffman tree.
    """
    # 1. Create a min-heap of leaf nodes
    heap = []
    for ch, f in freqs.items():
        heapq.heappush(heap, HuffmanNode(ch, f))

    # Edge case: if there is only one unique character
    if len(heap) == 1:
        only = heapq.heappop(heap)
        root = HuffmanNode(None, only.freq)
        root.left = only
        return root

    # 2. Combine the two smallest until one node remains
    while len(heap) > 1:
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)

        parent = HuffmanNode(None, left.freq + right.freq)
        parent.left = left
        parent.right = right

        heapq.heappush(heap, parent)

    # Remaining node is the root
    return heap[0]


def build_codes(root: HuffmanNode) -> Dict[str, str]:
    """
    Traverse the Huffman tree and produce a dict {character: code}.
    """
    codes: Dict[str, str] = {}

    def _dfs(node: HuffmanNode, current_code: str):
        if node is None:
            return
        # Leaf node: has a character
        if node.char is not None:
            # Edge case: if there is only one symbol, give it code "0"
            codes[node.char] = current_code or "0"
            return

        _dfs(node.left, current_code + "0")
        _dfs(node.right, current_code + "1")

    _dfs(root, "")
    return codes


def huffman_encode(text: str) -> Tuple[str, Dict[str, str], HuffmanNode]:
    """
    Given a text, build frequencies, Huffman tree, and encode the text.
    Returns:
      - encoded bitstring
      - codes dictionary
      - root of the Huffman tree
    """
    # Build frequency table
    freqs: Dict[str, int] = {}
    for ch in text:
        freqs[ch] = freqs.get(ch, 0) + 1

    # Build tree and codes
    root = build_huffman_tree(freqs)
    codes = build_codes(root)

    # Encode text
    encoded = "".join(codes[ch] for ch in text)
    return encoded, codes, root


def huffman_decode(encoded: str, root: HuffmanNode) -> str:
    """
    Decode a Huffman-encoded bitstring using the Huffman tree root.
    """
    result = []
    node = root

    for bit in encoded:
        node = node.left if bit == "0" else node.right

        # Leaf node → append character
        if node.char is not None:
            result.append(node.char)
            node = root

    return "".join(result)


if __name__ == "__main__":
    # --- Example 1: directly from a text string ---
    text = "huffman coding example"

    print("Original text:", text)

    encoded, codes, root = huffman_encode(text)

    print("\nHuffman Codes:")
    for ch, code in codes.items():
        # Show spaces clearly
        display = repr(ch)
        print(f"{display}: {code}")

    print("\nEncoded bitstring:", encoded)
    decoded = huffman_decode(encoded, root)
    print("Decoded text:", decoded)

    # --- Example 2: given explicit frequencies (set of characters) ---
    # You can also specify characters and frequencies manually:
    freqs = {
        'a': 5,
        'b': 9,
        'c': 12,
        'd': 13,
        'e': 16,
        'f': 45
    }

    root2 = build_huffman_tree(freqs)
    codes2 = build_codes(root2)

    print("\nHuffman codes for explicit frequencies:")
    for ch, code in codes2.items():
        print(f"{ch}: {code}")


Original text: huffman coding example

Huffman Codes:
'a': 000
'f': 001
'n': 010
'u': 0110
'd': 0111
'e': 100
' ': 1010
'm': 1011
'h': 11000
'g': 11001
'x': 11010
'p': 11011
'i': 11100
'c': 11101
'o': 11110
'l': 11111

Encoded bitstring: 1100001100010011011000010101011101111100111111000101100110101001101000010111101111111100
Decoded text: huffman coding example

Huffman codes for explicit frequencies:
f: 0
c: 100
d: 101
a: 1100
b: 1101
e: 111


When we ran the Huffman coding program on my sample text, it first read the input and counted how many times each character appeared, then used these frequencies to build a Huffman tree and generate a unique binary code for every character. Characters that occurred more frequently (for example the space character or common letters like ‘e’ and ‘a’) were assigned very short codes, while rare characters were given longer codes. Using these variable-length codes, the program then encoded the entire text into a compressed bitstring and finally decoded it back to the original message without any loss of information. This demonstrates the core idea of Huffman coding: by giving shorter codes to frequent symbols and longer codes to infrequent ones, the overall encoded message becomes smaller than if we used fixed-length codes (such as 8-bit ASCII for every character), while still allowing perfect reconstruction of the original text.